# Module M2.1 — Comprehensive Exploratory Data Analysis

**Project:** Explainable and Bias-Aware ML for Phishing Website Detection  
**Roadmap ref:** Phase 2 → Module M2.1  

### Sections
1. Environment Setup  
2. Dataset Loading  
3. Dataset Overview  
4. Distribution Analysis  
5. Correlation Analysis  
6. Leakage Analysis  
7. TLD Analysis  
8. Feature Importance Pre-Screening  
9. Outlier Analysis  
10. Report Generation  
11. Conclusions  


## 0. Environment Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


Project root: C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.logger              import get_logger
from src.features.feature_catalog  import get_feature_lists
from src.eda.visualization_manager import (
    setup_plot_style, save_figure, generate_eda_report
)
from src.eda.eda_analyzer     import run_eda_overview
from src.eda.correlation_analyzer import run_correlation_analysis
from src.eda.distribution_analyzer import run_distribution_analysis
from src.eda.leakage_analyzer import run_leakage_analysis

logger = get_logger('notebook.03_eda')
setup_plot_style()
print('Imports OK ✓')


Imports OK ✓


## 1. Path Configuration

In [3]:
CLEAN_CSV   = PROJECT_ROOT / 'data' / 'processed' / 'clean_df.csv'
REPORT_DIR  = PROJECT_ROOT / 'outputs' / 'reports'
PLOTS_EDA   = PROJECT_ROOT / 'outputs' / 'plots' / 'eda'

assert CLEAN_CSV.exists(), f'Run M1.1 first — {CLEAN_CSV} not found'

REPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_EDA.mkdir(parents=True, exist_ok=True)
(PLOTS_EDA / 'distributions').mkdir(parents=True, exist_ok=True)

print(f'Clean CSV  : {CLEAN_CSV}')
print(f'Reports    : {REPORT_DIR}')
print(f'Plots      : {PLOTS_EDA}')


Clean CSV  : C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\data\processed\clean_df.csv
Reports    : C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\outputs\reports
Plots      : C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\outputs\plots\eda


## 2. Dataset Loading

In [4]:
df = pd.read_csv(CLEAN_CSV, low_memory=False)
fl = get_feature_lists()

print(f'Shape         : {df.shape}')
print(f'Track B feats : {len(fl.track_B)}')
print(f'Track A feats : {len(fl.track_A)}')
print(f'Target col    : {fl.target}')
display(df.head(3))


Shape         : (235370, 56)
Track B feats : 49
Track A feats : 50
Target col    : label


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1


## 3. Dataset Overview

In [5]:
from src.eda.eda_analyzer import (
    compute_dataset_overview, save_overview_csv,
    plot_class_distribution, plot_dtype_summary,
)

overview = compute_dataset_overview(df, fl.track_B)
save_overview_csv(overview, REPORT_DIR / 'dataset_overview.csv')

cd = overview['class_distribution']
print(f"Rows          : {overview['n_rows']:,}")
print(f"Track B feat  : {overview['n_features']}")
print(f"Missing values: {overview['total_missing']}")
print(f"Memory        : {overview['memory_mb']:.1f} MB")
print(f"Phishing (0)  : {cd['phishing_count']:,}  ({cd['phishing_pct']:.2f}%)")
print(f"Legitimate (1): {cd['legitimate_count']:,}  ({cd['legitimate_pct']:.2f}%)")
print(f"Imbalance     : {cd['imbalance_ratio']:.4f}")
print('Saved → outputs/reports/dataset_overview.csv')


2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     | Computing dataset overview …
2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     |   Rows        : 235,370
2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     |   Track B feat: 49
2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     |   Missing     : 0
2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     |   Memory      : 183.3 MB
2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     |   Phishing    : 100,520 (42.71%)
2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     |   Legitimate  : 134,850 (57.29%)
2026-06-17 20:20:50 | INFO     | src.eda.eda_analyzer                     | Saved: dataset_overview.csv


Rows          : 235,370
Track B feat  : 49
Missing values: 0
Memory        : 183.3 MB
Phishing (0)  : 100,520  (42.71%)
Legitimate (1): 134,850  (57.29%)
Imbalance     : 0.7454
Saved → outputs/reports/dataset_overview.csv


In [6]:
plot_class_distribution(df, PLOTS_EDA)
plot_dtype_summary(df, fl.track_B, PLOTS_EDA)
print('Plots saved ✓')


findfont: Failed to find font weight 500, now using 400.
findfont: Failed to find font weight 600, now using 700.


Plots saved ✓


In [7]:
# Feature category summary
cat_summary = []
for cat, feats in fl.categories.items():
    num_in_cat = sum(1 for f in feats
                     if df[f].dtype in ['int64','float64'])
    cat_summary.append({
        'category': cat,
        'n_features': len(feats),
        'n_numeric': num_in_cat,
        'n_binary_or_cat': len(feats) - num_in_cat,
    })
cat_df = pd.DataFrame(cat_summary)
print(cat_df.to_string(index=False))


                 category  n_features  n_numeric  n_binary_or_cat
            URL Structure           7          6                1
          URL Statistical           3          3                0
URL Character Composition           9          9                0
              Obfuscation           3          3                0
           HTML Structure           8          8                0
   Redirects & Navigation           4          4                0
      Forms & Interaction           4          4                0
          Content & Trust           5          5                0
       External Resources           6          6                0


## 4. Distribution Analysis

### 4a. Individual Feature Distributions

Each feature gets a 4-panel figure (histogram, KDE by class, boxplot, class-separated boxplot).
Set `PLOT_ALL_DISTS = True` to generate all 49 plots (takes ~2 minutes).


In [8]:
# Set to True to generate all 49 per-feature distribution plots
PLOT_ALL_DISTS = True

dist_results = run_distribution_analysis(
    df        = df,
    features  = fl.track_B,
    output_dir= REPORT_DIR,
    plots_dir = PLOTS_EDA,
    plot_all  = PLOT_ALL_DISTS,
)
print(f"Distribution plots generated: {len(dist_results['distribution_paths'])}")
print(f"Features with >5% outliers  : {dist_results['n_outlier_features']}")


2026-06-17 20:20:50 | INFO     | src.eda.distribution_analyzer            | ============================================================
2026-06-17 20:20:50 | INFO     | src.eda.distribution_analyzer            | M2.1 — DISTRIBUTION ANALYSIS MODULE
2026-06-17 20:20:50 | INFO     | src.eda.distribution_analyzer            | ============================================================
2026-06-17 20:20:50 | INFO     | src.eda.distribution_analyzer            | Generating distribution plots for 49 features …
2026-06-17 20:21:56 | INFO     | src.eda.distribution_analyzer            | Distribution plots saved: 49


Distribution plots generated: 49
Features with >5% outliers  : 14


In [9]:
# Preview outlier stats table
outlier_df = dist_results['outlier_stats']
print('Top 10 features by outlier %:')
display(outlier_df[['feature','median','p99','max','outlier_pct']].head(10))


Top 10 features by outlier %:


,feature,median,p99,max,outlier_pct
0,NoOfEmptyRef,0.0,31.00,4887.0,16.5136
1,NoOfiFrame,0.0,21.00,1602.0,14.6310
2,URLLength,27.0,144.00,6097.0,9.5008
3,NoOfExternalRef,10.0,423.00,27516.0,9.4770
4,NoOfLettersInURL,14.0,99.00,5191.0,8.3022
5,LineOfCode,431.0,10401.24,442666.0,8.1718
6,NoOfImage,9.0,244.00,8956.0,7.8013
7,NoOfCSS,2.0,51.00,35820.0,7.6076
8,LargestLineLength,1091.0,129251.00,13975732.0,7.4092
9,NoOfSelfRef,12.0,517.00,27397.0,6.6177


## 5. Correlation Analysis

In [10]:
corr_results = run_correlation_analysis(
    df        = df,
    features  = fl.track_B,
    output_dir= REPORT_DIR,
    plots_dir = PLOTS_EDA,
)
print(f"High-corr pairs (|r|≥0.75)   : {corr_results['n_high_pairs']}")
print(f"Network edges (|r|≥0.30)      : {len(corr_results['network'])}")


2026-06-17 20:22:02 | INFO     | src.eda.correlation_analyzer             | ============================================================
2026-06-17 20:22:02 | INFO     | src.eda.correlation_analyzer             | M2.1 — CORRELATION ANALYSIS MODULE
2026-06-17 20:22:02 | INFO     | src.eda.correlation_analyzer             | ============================================================
2026-06-17 20:22:02 | INFO     | src.eda.correlation_analyzer             | Computing Pearson correlation (48 numeric features) …
2026-06-17 20:22:03 | INFO     | src.eda.correlation_analyzer             | Pearson matrix computed ✓
2026-06-17 20:22:03 | INFO     | src.eda.correlation_analyzer             | Computing Spearman correlation (48 features, 50,000 rows) …
2026-06-17 20:22:04 | INFO     | src.eda.correlation_analyzer             | Spearman matrix computed ✓
2026-06-17 20:22:04 | INFO     | src.eda.correlation_analyzer             | Top-20 positive pairs extracted (max r=0.9560)
2026-06-17 20:22:04 |

High-corr pairs (|r|≥0.75)   : 9
Network edges (|r|≥0.30)      : 129


In [11]:
print('Top 10 positive correlations (Pearson):')
display(corr_results['top_positive'][['feat_A','feat_B','pearson_r']].head(10))

print('\nTop 10 negative correlations (Pearson):')
display(corr_results['top_negative'][['feat_A','feat_B','pearson_r']].head(10))


Top 10 positive correlations (Pearson):


,feat_A,feat_B,pearson_r
0,URLLength,NoOfLettersInURL,0.956046
1,URLLength,NoOfDegitsInURL,0.835966
2,NoOfDegitsInURL,NoOfEqualsInURL,0.806379
3,HasObfuscation,ObfuscationRatio,0.798787
4,NoOfAmpersandInURL,NoOfObfuscatedChar,0.786460
5,NoOfEqualsInURL,NoOfOtherSpecialCharsInURL,0.785430
6,URLLength,NoOfOtherSpecialCharsInURL,0.782596
7,NoOfDegitsInURL,NoOfOtherSpecialCharsInURL,0.767919
8,NoOfEqualsInURL,NoOfObfuscatedChar,0.754563
9,NoOfDegitsInURL,NoOfObfuscatedChar,0.721518



Top 10 negative correlations (Pearson):


,feat_A,feat_B,pearson_r
0,CharContinuationRate,SpacialCharRatioInURL,-0.711704
1,URLCharProb,DegitRatioInURL,-0.708478
2,SpacialCharRatioInURL,DomainTitleMatchScore,-0.587548
3,DomainLength,CharContinuationRate,-0.576411
4,NoOfSubDomain,CharContinuationRate,-0.482611
5,SpacialCharRatioInURL,HasSocialNet,-0.429500
6,CharContinuationRate,NoOfOtherSpecialCharsInURL,-0.425095
7,URLCharProb,SpacialCharRatioInURL,-0.396971
8,SpacialCharRatioInURL,HasCopyrightInfo,-0.384272
9,SpacialCharRatioInURL,HasDescription,-0.378910


In [12]:
# Inspect correlation matrix snippet
print('Pearson matrix shape:', corr_results['pearson_matrix'].shape)
print('Spearman matrix shape:', corr_results['spearman_matrix'].shape)

# Show correlation with target
target_corr = df[fl.track_B + ['label']].select_dtypes(include=[np.number]).corr()['label']
target_corr = target_corr.drop('label').abs().sort_values(ascending=False)
print('\nTop 10 features correlated with label (Track B):')
print(target_corr.head(10).to_string())


Pearson matrix shape: (48, 48)
Spearman matrix shape: (48, 48)

Top 10 features correlated with label (Track B):
HasSocialNet             0.783882
HasCopyrightInfo         0.743197
HasDescription           0.690011
IsHTTPS                  0.610253
DomainTitleMatchScore    0.584204
HasSubmitButton          0.578816
IsResponsive             0.548977
SpacialCharRatioInURL    0.533003
HasHiddenFields          0.507715
HasFavicon               0.493607


## 6. Leakage Analysis

In [13]:
leakage_results = run_leakage_analysis(
    df        = df,
    output_dir= REPORT_DIR,
    plots_dir = PLOTS_EDA,
)

usi   = leakage_results['urlsimilarity']
https = leakage_results['https']

print('=== URLSimilarityIndex ===')
print(f"  All legitimate = 100.0 : {usi['all_legit_100']}")
print(f"  Phishing mean          : {usi['phishing_mean']}")
print(f"  Legitimate mean        : {usi['legit_mean']}")
print(f"  AUROC (solo)           : {usi['auroc']:.6f}")

print('\n=== IsHTTPS ===')
print(f"  All legitimate HTTPS=1 : {https['all_legit_https1']}")
print(f"  Phishing HTTPS rate    : {https['phishing_https_pct']:.1f}%")
print(f"  Legitimate HTTPS rate  : {https['legit_https_pct']:.1f}%")
print(f"  AUROC (solo)           : {https['auroc']:.6f}")


2026-06-17 20:22:13 | INFO     | src.eda.leakage_analyzer                 | ============================================================
2026-06-17 20:22:13 | INFO     | src.eda.leakage_analyzer                 | M2.1 — LEAKAGE ANALYSIS MODULE
2026-06-17 20:22:13 | INFO     | src.eda.leakage_analyzer                 | ============================================================
2026-06-17 20:22:13 | INFO     | src.eda.leakage_analyzer                 | URLSimilarityIndex AUROC          : 0.996090
2026-06-17 20:22:13 | INFO     | src.eda.leakage_analyzer                 |   All legitimate = 100.0          : True
2026-06-17 20:22:13 | INFO     | src.eda.leakage_analyzer                 |   Phishing mean                   : 49.6515
2026-06-17 20:22:13 | INFO     | src.eda.leakage_analyzer                 |   Legitimate mean                 : 100.0
2026-06-17 20:22:18 | INFO     | src.eda.leakage_analyzer                 | IsHTTPS AUROC                     : 0.754387
2026-06-17 20:22:18 | 

=== URLSimilarityIndex ===
  All legitimate = 100.0 : True
  Phishing mean          : 49.6515
  Legitimate mean        : 100.0
  AUROC (solo)           : 0.996090

=== IsHTTPS ===
  All legitimate HTTPS=1 : True
  Phishing HTTPS rate    : 49.1%
  Legitimate HTTPS rate  : 100.0%
  AUROC (solo)           : 0.754387


In [14]:
# Validation assertions
assert usi['all_legit_100'], 'URLSimilarityIndex leakage not confirmed'
assert usi['auroc'] > 0.95,  'URLSimilarityIndex AUROC unexpectedly low'
assert https['all_legit_https1'], 'IsHTTPS not all-1 for legitimate'
print('All leakage assertions PASSED ✓')


All leakage assertions PASSED ✓


## 7. TLD Analysis

In [15]:
from src.eda.eda_analyzer import compute_tld_analysis, plot_tld_phishing_rate, plot_tld_frequency

tld_results = compute_tld_analysis(df)

print(f"Unique TLDs      : {tld_results['n_unique_tlds']}")
print(f"Long-tail TLDs   : {tld_results['long_tail_count']} (< 100 samples)")

top_tlds = tld_results['top_tlds']
print('\nTop 20 TLDs by sample count:')
display(top_tlds[['TLD','count','phishing_count','phishing_rate']].head(20))


2026-06-17 20:22:19 | INFO     | src.eda.eda_analyzer                     | Computing TLD analysis …
2026-06-17 20:22:19 | INFO     | src.eda.eda_analyzer                     |   Unique TLDs    : 695
2026-06-17 20:22:19 | INFO     | src.eda.eda_analyzer                     |   Long-tail TLDs : 593 (< 100 samples)
2026-06-17 20:22:19 | INFO     | src.eda.eda_analyzer                     |   Top TLD        : com  (112,382 samples)


Unique TLDs      : 695
Long-tail TLDs   : 593 (< 100 samples)

Top 20 TLDs by sample count:


,TLD,count,phishing_count,phishing_rate
0,com,112382,43597,0.387936
1,org,18792,2268,0.120690
2,net,7076,3078,0.434992
3,app,6467,6327,0.978352
4,uk,6395,322,0.050352
5,co,5408,4950,0.915311
6,io,4174,3742,0.896502
7,de,3994,684,0.171257
8,ru,3870,2978,0.769509
9,au,2979,373,0.125210


In [16]:
# Most phishing-heavy TLDs (min 100 samples)
high_phish = top_tlds[top_tlds['count'] >= 100].nlargest(10, 'phishing_rate')
print('Most phishing-heavy TLDs (min 100 samples):')
display(high_phish[['TLD','count','phishing_rate']].reset_index(drop=True))

# Safest TLDs
safest = top_tlds[top_tlds['count'] >= 100].nsmallest(10, 'phishing_rate')
print('\nSafest TLDs (min 100 samples):')
display(safest[['TLD','count','phishing_rate']].reset_index(drop=True))


Most phishing-heavy TLDs (min 100 samples):


,TLD,count,phishing_rate
0,cf,1203,1.000000
1,gq,493,1.000000
2,page,489,1.000000
3,icu,150,1.000000
4,cfd,107,1.000000
5,top,2327,0.999141
6,ml,994,0.998994
7,site,1470,0.996599
8,ga,1111,0.996400
9,link,1433,0.994417



Safest TLDs (min 100 samples):


,TLD,count,phishing_rate
0,gov,1276,0.000000
1,mil,301,0.000000
2,edu,1861,0.002687
3,ie,468,0.019231
4,no,402,0.032338
5,bg,137,0.043796
6,hr,173,0.046243
7,uk,6395,0.050352
8,fi,323,0.058824
9,jp,2219,0.061740


In [17]:
plot_tld_phishing_rate(tld_results, PLOTS_EDA)
plot_tld_frequency(tld_results, PLOTS_EDA)
print('TLD plots saved ✓')


TLD plots saved ✓


## 8. Feature Importance Pre-Screening

In [18]:
from src.eda.eda_analyzer import (
    compute_feature_prescreening, plot_mutual_information, plot_anova_prescreening
)

ps_results = compute_feature_prescreening(df, fl.track_B, sample_n=50_000)

mi_df = ps_results['mutual_information']
print(f"Pre-screening computed for {ps_results['n_numeric']} numeric features")
print('\nTop 15 features by Mutual Information:')
display(mi_df[['feature','mi_score','anova_f','mi_rank']].head(15))


2026-06-17 20:22:21 | INFO     | src.eda.eda_analyzer                     | Computing feature pre-screening (sample_n=50,000) …
2026-06-17 20:22:32 | INFO     | src.eda.eda_analyzer                     |   MI computed for 48 numeric features
2026-06-17 20:22:32 | INFO     | src.eda.eda_analyzer                     |   Top-5 MI features: ['LineOfCode', 'NoOfExternalRef', 'NoOfImage', 'NoOfSelfRef', 'NoOfJS']
2026-06-17 20:22:32 | INFO     | src.eda.eda_analyzer                     |   Chi-Square binary features: 19


Pre-screening computed for 48 numeric features

Top 15 features by Mutual Information:


,feature,mi_score,anova_f,mi_rank
0,LineOfCode,0.598895,6912.140688,1
1,NoOfExternalRef,0.559054,2854.051462,2
2,NoOfImage,0.541486,4722.791629,3
3,NoOfSelfRef,0.526191,5166.226810,4
4,NoOfJS,0.498724,10546.411588,5
5,LargestLineLength,0.474643,80.543683,6
6,NoOfCSS,0.445404,11499.367203,7
7,HasSocialNet,0.389937,78939.139188,8
8,LetterRatioInURL,0.380879,8240.876706,9
9,HasCopyrightInfo,0.324758,61392.190254,10


In [19]:
plot_mutual_information(ps_results, PLOTS_EDA)
plot_anova_prescreening(ps_results, PLOTS_EDA)
print('Pre-screening plots saved ✓')


Pre-screening plots saved ✓


## 9. Outlier Analysis

In [20]:
from src.eda.distribution_analyzer import (
    OUTLIER_FEATURES, compute_outlier_stats, plot_outlier_boxplots, plot_skewness_summary
)

valid_outlier_feats = [f for f in OUTLIER_FEATURES if f in df.columns]
outlier_df = compute_outlier_stats(df, valid_outlier_feats)

print('Full Outlier Statistics:')
display(outlier_df[['feature','q1','median','q3','p99','max','n_outliers','outlier_pct']])


Full Outlier Statistics:


,feature,q1,median,q3,p99,max,n_outliers,outlier_pct
0,NoOfEmptyRef,0.0,0.0,1.0,31.00,4887.0,38868,16.5136
1,NoOfiFrame,0.0,0.0,1.0,21.00,1602.0,34437,14.6310
2,URLLength,23.0,27.0,34.0,144.00,6097.0,22362,9.5008
3,NoOfExternalRef,1.0,10.0,58.0,423.00,27516.0,22306,9.4770
4,NoOfLettersInURL,10.0,14.0,20.0,99.00,5191.0,19541,8.3022
5,LineOfCode,18.0,431.0,1279.0,10401.24,442666.0,19234,8.1718
6,NoOfImage,0.0,9.0,29.0,244.00,8956.0,18362,7.8013
7,NoOfCSS,0.0,2.0,8.0,51.00,35820.0,17906,7.6076
8,LargestLineLength,200.0,1091.0,8047.0,129251.00,13975732.0,17439,7.4092
9,NoOfSelfRef,0.0,12.0,88.0,517.00,27397.0,15576,6.6177


In [21]:
plot_outlier_boxplots(df, valid_outlier_feats, PLOTS_EDA)
plot_skewness_summary(df, fl.track_B, PLOTS_EDA)
print('Outlier plots saved ✓')


Outlier plots saved ✓


In [22]:
# Identify features requiring log1p transformation (|skew| > 5)
num_feats = [c for c in fl.track_B if df[c].dtype in ['int64','float64']]
skew_vals = df[num_feats].skew().abs().sort_values(ascending=False)
log_candidates = skew_vals[skew_vals > 5]
print(f'Features needing log1p transform (|skew|>5): {len(log_candidates)}')
print(log_candidates.to_string())


Features needing log1p transform (|skew|>5): 22
NoOfCSS                       463.992297
NoOfObfuscatedChar            204.506366
NoOfJS                        140.375265
NoOfEqualsInURL               114.887278
NoOfEmptyRef                  106.738379
NoOfAmpersandInURL            106.627680
NoOfiFrame                     97.607158
NoOfDegitsInURL                94.907962
NoOfPopup                      84.879864
NoOfExternalRef                65.855061
NoOfSelfRef                    60.346935
NoOfLettersInURL               58.452964
URLLength                      53.346618
LineOfCode                     53.036533
LargestLineLength              47.851201
NoOfOtherSpecialCharsInURL     47.515701
ObfuscationRatio               40.140604
NoOfImage                      28.210672
HasObfuscation                 22.007197
IsDomainIP                     19.159489
NoOfQMarkInURL                  8.159639
Crypto                          6.292428


## 10. Complete Pipeline Run & HTML Report

In [23]:
# Collect all results for the HTML report
all_results = {
    'overview'    : overview,
    'leakage'     : leakage_results,
    'correlation' : corr_results,
    'tld'         : tld_results,
    'outliers'    : {
        'outlier_stats'       : outlier_df,
        'n_outlier_features'  : int((outlier_df['outlier_pct'] > 5).sum()),
    },
    'prescreening': ps_results,
}

report_path = generate_eda_report(
    results     = all_results,
    output_path = REPORT_DIR / 'm2_1_eda_report.html',
    plots_dir   = PLOTS_EDA,
)
print(f'EDA report saved: {report_path}')


2026-06-17 20:22:38 | INFO     | src.eda.visualization_manager            | M2.1 EDA report saved: C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\outputs\reports\m2_1_eda_report.html


EDA report saved: C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\outputs\reports\m2_1_eda_report.html


## 11. Conclusions & Preprocessing Recommendations

In [24]:
print('=' * 65)
print('MODULE M2.1 — EDA CONCLUSIONS')
print('=' * 65)

usi   = leakage_results['urlsimilarity']
https = leakage_results['https']
n_log = len(df[num_feats].skew()[df[num_feats].skew().abs() > 5])

print(f'  LEAKAGE  URLSimilarityIndex AUROC     : {usi["auroc"]:.4f} → exclude from Track B')
print(f'  ADVISORY IsHTTPS AUROC                : {https["auroc"]:.4f} → retain, flag for bias')
print(f'  CORR     High-corr pairs (|r|≥0.75)  : {corr_results["n_high_pairs"]}')
print(f'  TLD      Unique TLDs                  : {tld_results["n_unique_tlds"]} → frequency encode')
print(f'  OUTLIERS Features needing log1p        : {n_log}')
print(f'  IMBALANCE Phishing %                   : {overview["class_distribution"]["phishing_pct"]:.2f}% → mild, stratified CV')
print()
print('PREPROCESSING REQUIREMENTS FOR M3.1:')
print('  1. TLD frequency encoding (top-50 + rare_tld)')
print('  2. Log1p transform for all high-skew count features')
print('  3. Outlier capping at P99.9 before scaling')
print('  4. RobustScaler for continuous features')
print('  5. No imputation required (zero missing values)')
print('  6. Stratified 80/20 split (class balance preserved)')
print()
print('M2.1 COMPLETE. Next: M3.1 — Preprocessing Pipeline')


MODULE M2.1 — EDA CONCLUSIONS
  LEAKAGE  URLSimilarityIndex AUROC     : 0.9961 → exclude from Track B
  ADVISORY IsHTTPS AUROC                : 0.7544 → retain, flag for bias
  CORR     High-corr pairs (|r|≥0.75)  : 9
  TLD      Unique TLDs                  : 695 → frequency encode
  OUTLIERS Features needing log1p        : 22
  IMBALANCE Phishing %                   : 42.71% → mild, stratified CV

PREPROCESSING REQUIREMENTS FOR M3.1:
  1. TLD frequency encoding (top-50 + rare_tld)
  2. Log1p transform for all high-skew count features
  3. Outlier capping at P99.9 before scaling
  4. RobustScaler for continuous features
  5. No imputation required (zero missing values)
  6. Stratified 80/20 split (class balance preserved)

M2.1 COMPLETE. Next: M3.1 — Preprocessing Pipeline
